
# Plot DJF ENSO Cross-Section Regressions

Load cached Indonesia-domain longitude-pressure and latitude-pressure DJF Niño3.4 regression slopes, then render shading-only cross-sections with shared colorbars.


# 1. Setup


In [ ]:

from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from scipy.ndimage import gaussian_filter

plt.rcParams.update({
    'font.family': 'Helvetica',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
    'axes.titleweight': 'regular',
    'figure.dpi': 120,
})

PROJECT_ROOT = Path('/Users/rizzie/TugasAkhir/data_processing')
CACHE_DIR = PROJECT_ROOT / 'data/intermediate/divided_regression_1991_2020'
RESULTS_DIR = PROJECT_ROOT / 'results/analisis_regresi_26-19'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

LON_CACHE = CACHE_DIR / 'reg_xsec_lon_pressure_q_u_omega.nc'
LAT_CACHE = CACHE_DIR / 'reg_xsec_lat_pressure_q_v_omega.nc'

LON_BAND = slice(95.0, 125.0)
LAT_BAND = slice(-10.0, 5.0)
PRESSURE_TICKS = [1000, 925, 850, 700, 500, 400, 300, 200, 100]
LON_TICKS = np.arange(95, 130, 5)
LAT_TICKS = np.arange(-10, 6, 2)
VECTOR_SMOOTH_SIGMA = 0.6
SHADING_LEVELS = 21


def format_lon(value, pos=None):
    return f'{value:.0f}E'


def format_lat(value, pos=None):
    if np.isclose(value, 0):
        return '0'
    return f"{abs(value):.0f}{'N' if value > 0 else 'S'}"


def load_cache(path):
    return xr.open_dataset(path).load()


def smooth_2d(values, sigma=VECTOR_SMOOTH_SIGMA):
    if not np.isfinite(values).any():
        return values
    finite = np.isfinite(values)
    fill_value = float(np.nanmean(values[finite])) if finite.any() else 0.0
    filled = np.where(finite, values, fill_value)
    return gaussian_filter(filled, sigma=sigma)


def symmetric_levels(values, n=SHADING_LEVELS):
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return np.linspace(-1, 1, n)
    vmax = float(np.nanpercentile(np.abs(finite), 98))
    if not np.isfinite(vmax) or vmax == 0:
        vmax = float(np.nanmax(np.abs(finite))) if finite.size else 1.0
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0
    return np.linspace(-vmax, vmax, n)


def render_cross_section_grid(top_dict, bottom_dict, periods, title, cbar_label, out_name, value_scale=1.0, cbar_ticks=None, cbar_power=None):
    fig, axes = plt.subplots(2, 3, figsize=(14.5, 7.8), sharey=True)

    all_values = []
    for period in periods:
        all_values.append(np.asarray(top_dict[period].values).ravel())
        all_values.append(np.asarray(bottom_dict[period].values).ravel())
    all_values = np.concatenate(all_values) * value_scale
    levels = symmetric_levels(all_values)
    ticks = np.linspace(levels[0], levels[-1], 7)
    if cbar_ticks is None:
        cbar_ticks = ticks

    row_specs = [
        ('Longitude-pressure', top_dict, LON_BAND, 'Longitude (°E)', LON_TICKS, format_lon),
        ('Latitude-pressure', bottom_dict, LAT_BAND, 'Latitude (°N)', LAT_TICKS, format_lat),
    ]
    period_labels = {
        'p1': 'P1 1981-2006',
        'p2': 'P2 2007-2025',
        'delta': 'P2-P1',
    }

    mesh = None
    for row, (row_label, row_dict, x_band, x_label, x_ticks, formatter) in enumerate(row_specs):
        for col, period in enumerate(periods):
            ax = axes[row, col]
            da = row_dict[period]
            x_dim = 'lon' if 'lon' in da.dims else 'lat'
            da2 = da.transpose('level', x_dim)
            values = smooth_2d(da2.values) * value_scale
            x = da2[x_dim].values
            y = da2['level'].values

            mesh = ax.contourf(x, y, values, levels=levels, cmap='RdBu_r', extend='both')
            ax.contour(x, y, values, levels=ticks, colors='k', linewidths=0.35, alpha=0.25)

            ax.set_xlim(x_band.start, x_band.stop)
            ax.set_ylim(1000, 100)
            ax.set_xticks(x_ticks)
            ax.set_yticks(PRESSURE_TICKS)
            ax.xaxis.set_major_formatter(FuncFormatter(formatter))
            ax.tick_params(axis='both', labelsize=10)
            ax.grid(True, linestyle='--', alpha=0.18, linewidth=0.45)
            if row == 1:
                ax.set_xlabel(x_label, fontsize=12, labelpad=8)
            else:
                ax.set_xlabel('')
            if col == 0:
                ax.set_ylabel('Pressure level (hPa)', fontsize=12, labelpad=10)
            else:
                ax.set_ylabel('')
            ax.set_title(period_labels[period], fontsize=12, fontweight='bold', pad=8)
            ax.text(
                0.02,
                0.05,
                row_label,
                transform=ax.transAxes,
                fontsize=10,
                fontweight='bold',
                ha='left',
                va='bottom',
                bbox={'facecolor': 'white', 'alpha': 0.6, 'edgecolor': 'none'},
            )

    fig.suptitle(title, fontsize=15, fontweight='bold', y=0.99)
    fig.text(0.015, 0.74, 'Longitude-pressure', rotation=90, va='center', ha='center', fontsize=12, fontweight='bold')
    fig.text(0.015, 0.28, 'Latitude-pressure', rotation=90, va='center', ha='center', fontsize=12, fontweight='bold')
    fig.subplots_adjust(left=0.07, right=0.90, top=0.92, bottom=0.08, wspace=0.15, hspace=0.16)
    cbar = fig.colorbar(mesh, ax=axes.ravel().tolist(), pad=0.02, shrink=0.96, ticks=cbar_ticks)
    if cbar_power is not None:
        cbar_label = f'{cbar_label} (×10^{cbar_power})'
    cbar.set_label(cbar_label, fontsize=12, labelpad=10)
    if cbar_power is not None:
        cbar.ax.yaxis.set_major_formatter(FuncFormatter(lambda v, pos=None: f'{v:.0f}'))
    cbar.ax.tick_params(labelsize=10)
    fig.savefig(RESULTS_DIR / out_name, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)


# 2. Load Cached Regressions


In [ ]:

lon_ds = load_cache(LON_CACHE)
lat_ds = load_cache(LAT_CACHE)

lon_q = {'p1': lon_ds['q_reg_p1'], 'p2': lon_ds['q_reg_p2'], 'delta': lon_ds['q_reg_delta']}
lon_omega = {'p1': lon_ds['omega_reg_p1'], 'p2': lon_ds['omega_reg_p2'], 'delta': lon_ds['omega_reg_delta']}

lat_q = {'p1': lat_ds['q_reg_p1'], 'p2': lat_ds['q_reg_p2'], 'delta': lat_ds['q_reg_delta']}
lat_omega = {'p1': lat_ds['omega_reg_p1'], 'p2': lat_ds['omega_reg_p2'], 'delta': lat_ds['omega_reg_delta']}

print('lon regression cache vars:', list(lon_ds.data_vars))
print('lat regression cache vars:', list(lat_ds.data_vars))



# 3. Q Regression Shading


In [ ]:

render_cross_section_grid(
    lon_q,
    lat_q,
    ['p1', 'p2', 'delta'],
    'q regression anomaly vs Niño3.4',
    'q regression slope per 1°C Niño3.4',
    'reg_xsec_q_grid.png',
    value_scale=1e4,
    cbar_ticks=np.arange(-3, 4, 1),
    cbar_power=-4,
)



# 4. Omega Regression Shading


In [ ]:

render_cross_section_grid(
    lon_omega,
    lat_omega,
    ['p1', 'p2', 'delta'],
    'omega regression anomaly vs Niño3.4',
    'omega regression slope per 1°C Niño3.4',
    'reg_xsec_omega_grid.png',
)



# 5. Revision Summary

- Replaced the vector-style cross-section plot with shading-only 2x3 grids.
- Longitude-pressure is the top row and latitude-pressure is the bottom row.
- q and omega are plotted separately, each with a shared colorbar across the full grid.
- The notebook now reads only the cached regression fields needed for shading.
